# AutoGen Demo 3: Multi-Agent Collaboration and Planning

In the previous demo, agents challenged each other through structured debate.

In this demo, the agents are no longer competing.  
Instead, they collaborate to solve a shared business problem through specialization and iterative refinement.

This introduces one of the most important ideas in agentic systems:

> Different agents can contribute different expertise toward a shared goal.

The objective is not just to generate text.  
The objective is to improve a shared artifact over time.

---

## Why This Matters

Single-agent prompting often produces:
- generic recommendations
- shallow prioritization
- weak operational realism

Collaborative multi-agent systems can improve outcomes by:
- separating responsibilities
- introducing critique and constraints
- refining ideas iteratively
- synthesizing multiple perspectives

This resembles real-world business and engineering workflows:
- marketing teams optimize awareness
- operations teams manage feasibility
- finance teams manage risk and budgets
- leadership synthesizes tradeoffs into a final strategy

The interaction between specialized roles becomes part of the planning process.

---

## Scenario

A client runs a bicycle repair business out of his garage in Austin, Texas.

Current situation:
- He has the tools and experience needed to operate successfully.
- The business is run by one person.
- He has limited marketing experience.
- He wants to increase awareness and revenue over the next 3 months.
- He has a maximum promotional budget of $2500.
- Expensive channels like TV, radio, and billboards are not feasible.
- Austin is entering peak cycling season, with several local cycling events expected soon.
- The owner can only handle a moderate increase in workload without hiring help.

The agents must collaborate to produce a realistic growth strategy.

---

## Agent Roles

### MarketingAgent
Focuses on:
- local promotion
- awareness
- community engagement
- low-cost marketing strategies

### OperationsAgent
Focuses on:
- workload
- scheduling
- service capacity
- operational feasibility

### FinanceAgent
Focuses on:
- budget allocation
- return on investment
- sustainability
- financial risk

### StrategyAgent
Responsible for:
- synthesizing all recommendations
- resolving tradeoffs
- producing a coherent final plan

---

## What To Watch For

As the collaboration progresses, observe:

- How each agent contributes different expertise
- Whether agents meaningfully react to previous outputs
- Whether critique improves the plan
- How constraints shape the recommendations
- Whether the final strategy feels more realistic than the baseline

Also notice the tradeoffs:
- coordination increases complexity
- longer conversations increase token usage
- too many agents can create noise
- specialization only helps if the workflow is structured carefully

---

## Important Teaching Point

The benefit does not come from "more AI agents."

The benefit comes from:
- specialization
- structured collaboration
- iterative refinement
- constraint-aware planning

The conversation itself becomes part of the planning workflow.

In [1]:
# Install packages if needed:
# pip install "autogen-agentchat" "autogen-ext[openai]"

import os
import asyncio
from dataclasses import dataclass, field

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()


True

In [2]:
# Configure model client

model_client = OpenAIChatCompletionClient(
    model="gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)

In [3]:
BUSINESS_SCENARIO = """
A client runs a bicycle repair business out of his garage in Austin, Texas.

Current situation:
- He has the tools and experience needed to operate successfully.
- The business is run by one person.
- He has limited marketing experience.
- He wants to increase awareness and revenue over the next 3 months.
- He has a maximum promotional budget of $2500.
- Expensive channels like TV, radio, and billboards are not feasible.
- Austin is entering peak cycling season, with several local cycling events expected soon.
- The owner can only handle a moderate increase in workload without hiring help.
"""

In [4]:
# Define agents

marketing_agent = AssistantAgent(
    name="MarketingAgent",
    model_client=model_client,
    system_message="""
You are MarketingAgent.

Focus on:
- local awareness
- community engagement
- low-cost digital promotion
- cycling-related partnerships and events
- realistic small-business marketing

Rules:
- stay under 180 words
- provide practical recommendations
- avoid unrealistic scale
- directly respond to prior recommendations when appropriate
""",
)

operations_agent = AssistantAgent(
    name="OperationsAgent",
    model_client=model_client,
    system_message="""
You are OperationsAgent.

Focus on:
- operational feasibility
- scheduling
- customer throughput
- service turnaround
- workload constraints

Rules:
- stay under 180 words
- identify operational bottlenecks
- challenge unrealistic growth assumptions
- directly respond to prior recommendations when appropriate
""",
)

finance_agent = AssistantAgent(
    name="FinanceAgent",
    model_client=model_client,
    system_message="""
You are FinanceAgent.

Focus on:
- budget allocation
- ROI
- sustainability
- avoiding wasteful spending
- realistic financial planning

Rules:
- stay under 180 words
- challenge weak ROI assumptions
- prioritize cost-effective approaches
- directly respond to prior recommendations when appropriate
""",
)

strategy_agent = AssistantAgent(
    name="StrategyAgent",
    model_client=model_client,
    system_message="""
You are StrategyAgent.

Your job:
- synthesize all recommendations
- resolve tradeoffs
- prioritize actions
- produce a realistic final business strategy

Rules:
- stay concise and practical
- produce a coherent plan
- include a rough budget allocation
- include implementation priorities
""",
)

In [5]:
@dataclass
class CollaborationTurn:
    speaker: str
    message: str


@dataclass
class CollaborationState:
    turns: list[CollaborationTurn] = field(default_factory=list)

    def add_turn(self, speaker: str, message: str):
        self.turns.append(
            CollaborationTurn(speaker, message)
        )

    def transcript(self):
        return "\n\n".join(
            f"{turn.speaker}: {turn.message}"
            for turn in self.turns
        )

In [6]:
async def get_single_agent_baseline():
    baseline_agent = AssistantAgent(
        name="BaselineAgent",
        model_client=model_client,
        system_message="""
Provide practical small-business advice.
Stay concise and realistic.
""",
    )

    prompt = f"""
Business scenario:

{BUSINESS_SCENARIO}

Provide a practical business growth plan.
"""

    result = await baseline_agent.run(task=prompt)

    return result.messages[-1].content

In [7]:
async def run_agent_step(agent, state, scenario):
    transcript = state.transcript()

    prompt = f"""
Business scenario:

{scenario}

Current collaboration transcript:

{transcript}

Provide your contribution based on your role.
"""

    result = await agent.run(task=prompt)

    response = result.messages[-1].content.strip()

    state.add_turn(agent.name, response)

    return response

In [8]:
async def run_collaboration():
    state = CollaborationState()

    working_agents = [
        marketing_agent,
        operations_agent,
        finance_agent,
    ]

    print("=" * 80)
    print("MULTI-AGENT BUSINESS COLLABORATION")
    print("=" * 80)
    print()

    for agent in working_agents:

        response = await run_agent_step(
            agent=agent,
            state=state,
            scenario=BUSINESS_SCENARIO,
        )

        print(f"[{agent.name}]")
        display(Markdown(response))
        print()
        print("-" * 80)
        print()

    return state

In [9]:
async def get_final_strategy(state):
    transcript = state.transcript()

    prompt = f"""
Business scenario:

{BUSINESS_SCENARIO}

Collaboration transcript:

{transcript}

Produce:
1. Final recommended strategy
2. Prioritized action plan
3. Approximate budget allocation
4. Key operational considerations
"""

    result = await strategy_agent.run(task=prompt)

    return result.messages[-1].content

In [10]:
# Single-agent baseline

baseline = await get_single_agent_baseline()

print("=" * 80)
print("SINGLE-AGENT BASELINE")
print("=" * 80)
print()
display(Markdown(baseline))

SINGLE-AGENT BASELINE



1. **Leverage Local Cycling Events (Budget: $800)**  
- Sponsor or set up a repair booth at 2-3 upcoming local cycling events to get direct exposure to potential customers.  
- Prepare flyers and business cards to distribute.  
- Offer event-day discounts or quick tune-ups to attract attention.

2. **Optimize Online Presence (Budget: $400)**  
- Create or update a simple, mobile-friendly website with clear services, pricing, contact info, and customer reviews. Use affordable platforms like Wix or Squarespace.  
- Set up and optimize a Google My Business profile to appear in local searches and Google Maps.

3. **Targeted Social Media Marketing (Budget: $600)**  
- Use Facebook and Instagram ads focused on Austin cyclists aged 18-50; allocate $200/month for 3 months.  
- Join and engage in local cycling groups on social media to build awareness authentically.

4. **Referral and Loyalty Program (Budget: $200)**  
- Implement a simple referral program: e.g., “Refer a friend and get 10% off your next repair.”  
- Offer a loyalty punch card: e.g., “5 tune-ups and the 6th is free.”

5. **Time Management to Handle Increased Demand**  
- Limit booking slots to a manageable number per day to avoid burnout.  
- Prioritize profitable, quick services.  
- Consider sharing workload occasionally with a trusted freelance mechanic as a backup, paid per job, if needed.

6. **Local Partnerships and Flyers (Budget: $500)**  
- Partner with local bike shops, gyms, and cafes for flyer placement and mutual referrals.  
- Distribute flyers in popular biking spots and community boards.

**Summary Budget:**  
- Events & Sponsorship: $800  
- Website & Google My Business: $400  
- Social Media Ads: $600  
- Referral/Loyalty: $200  
- Flyers & Local Partnerships: $500  
**Total:** $2,500

Focus on direct local engagement and targeted digital marketing to grow awareness and revenue sustainably within workload limits.

In [11]:
# Multi-agent collaboration

collaboration_state = await run_collaboration()

MULTI-AGENT BUSINESS COLLABORATION

[MarketingAgent]


To boost your client's bicycle repair business efficiently and affordably, focus on these strategies:

1. **Leverage Local Cycling Events**: Partner with event organizers to offer on-site quick repairs or discounts to participants. This builds immediate visibility among engaged cyclists.

2. **Community Engagement via Social Media**: Create a Facebook page and Instagram profile showcasing before/after repairs, quick tips, and event involvement. Use local hashtags (#AustinCycling #ATXbiking) to increase reach.

3. **Google My Business Optimization**: Ensure the business appears in local searches with accurate information, photos, and active review solicitation to build trust.

4. **Collaborate with Local Bike Shops and Clubs**: Offer referral discounts or workshop sessions. These partnerships increase credibility and customer flow without heavy resource demand.

5. **Flyer Distribution at Local Spots**: Place simple, eye-catching flyers at bike shops, cafes near popular cycling routes, and community boards, maximizing low-cost local exposure.

Budget allocation example:
- Event partnerships: $800 (small sponsorship or setup costs)
- Social media promotion: $500 (boost posts targeting Austin cyclists)
- Flyers & printing: $200
- Google Ads basic campaigns/local SEO: $500
- Contingency: $500

Moderate workload focus ensures manageable growth aligned with his capacity.


--------------------------------------------------------------------------------

[OperationsAgent]


The proposed marketing plan is operationally sound but must carefully balance customer influx with a single-operator workload. 

Bottleneck risks:
- On-site event repairs may spike demand beyond the owner's capacity; limiting services to quick fixes at events and booking follow-ups offsite can prevent overload.
- Referral partnerships and workshops might increase steady traffic; scheduling workshops during low-demand periods or weekends can evenly spread workload.
- Social media and flyer interest could generate inquiries needing prompt response—streamlining communication with templates or set office hours is advisable.

Challenging growth assumptions:
- Rapid customer volume increases without backup staff risk degrading service quality and turnaround time.
- Offering discounts should be tiered or time-limited to avoid prolonged high workload.

Scheduling careful appointment management and setting daily repair caps will safeguard service standards. The client should mechanically monitor workload daily and adapt marketing efforts progressively, using slower channels first to gauge impact within capacity constraints.

Overall, the plan can increase awareness and revenue but must be aligned with realistic service throughput limits to avoid operational bottlenecks.


--------------------------------------------------------------------------------

[FinanceAgent]


Financially, the $2,500 budget allocation is generally sound but can be optimized for better ROI and risk mitigation.

1. **Event Partnerships ($800)**: Confirm exact sponsorship costs and prioritize events with the highest attendance or targeted cycling demographics. Negotiate in-kind promotions (e.g., logo display for repair discounts) to reduce out-of-pocket expense. Consider reallocating surplus funds here if other channels underperform.

2. **Social Media Promotion ($500)**: Paid boosts should be highly targeted; test $100 initially on ads with clear calls to action before scaling. Organic content and engagement efforts can reduce dependence on paid reach, saving budget.

3. **Flyers & Printing ($200)**: Keep flyer design simple and focus on placement at high-traffic cycling hubs. Avoid overprinting as excess flyers waste resources.

4. **Google Ads / Local SEO ($500)**: Monitor spend daily to avoid costly clicks with low conversion. Invest more in free local SEO (Google My Business) by incentivizing reviews since this has sustained impact at near-zero cost.

5. **Contingency ($500)**: Preserve this as a buffer for unexpected needs or to boost high-performing channels quickly.

Avoid spreading funds too thin across many channels. Prioritize measurable, low-cost, and scalable tactics, adjusting budget based on early performance to maximize financial efficiency and sustainable growth.


--------------------------------------------------------------------------------



In [12]:
# Final synthesized strategy

final_strategy = await get_final_strategy(collaboration_state)

print("=" * 80)
print("FINAL BUSINESS STRATEGY")
print("=" * 80)
print()
display(Markdown(final_strategy))

FINAL BUSINESS STRATEGY



**1. Final Recommended Strategy**

Focus on building local cycling community visibility and credibility through targeted, manageable marketing channels that fit the single-operator capacity and maximize ROI within a $2,500 budget. Specifically:

- Leverage upcoming Austin cycling events with selective partnerships to gain direct access to engaged cyclists while limiting workload through quick onsite fixes and scheduled follow-ups.
- Develop organic and lightly boosted social media presence targeting local cyclists to build steady interest and trust.
- Optimize Google My Business and encourage online reviews for sustainable local search traction.
- Build low-effort referral partnerships with local bike shops and clubs, using simple discount incentives.
- Deploy targeted flyer distribution at high-cyclist-traffic locations for cost-effective awareness boosts.

These combined will amplify visibility and revenue while controlling workload and operational risks.

---

**2. Prioritized Action Plan**

**Month 1: Setup & Soft Launch**

- Register and optimize Google My Business; add photos, services, hours, and request initial reviews.
- Create Facebook and Instagram profiles; start posting engaging before/after photos, repair tips, and event announcements.
- Design simple, eye-catching flyers; identify and get permission for key local distribution spots (bike shops, cafes).
- Identify 2-3 key upcoming cycling events; contact organizers to propose quick repair/support partnership with limited scope.
- Reach out to 2-3 local bike shops and cycling clubs to propose referral discounts or workshop collaborations, scheduling workshops off-peak.

**Month 2: Execute & Boost**

- Launch small-scale social media ad campaign with $100 test budget focusing on event participants and local cycling groups; monitor engagement.
- Begin flyer distribution aligned with event timing and high-traffic spots.
- Provide quick onsite fixes at events; limit number and duration per customer (e.g., free minor tune-ups or discounted basic repairs), funneling others to scheduled appointments.
- Host first referral workshop or partner event timed during a slower weekday or weekend slot.
- Actively encourage satisfied customers to leave online reviews.

**Month 3: Monitor & Scale**

- Analyze which channels drive actual bookings and handle workload well.
- Carefully increase social media ad spend up to $400 based on test results, prioritizing highest ROI posts.
- Consider small sponsorship enhancements for top events if workload capacity allows.
- Continue flyer refreshes and community engagement.
- Implement scheduling caps and use templated responses for customer communication for efficiency.
- Adjust workshop frequency or referral offers based on customer flow.

---

**3. Approximate Budget Allocation**

| Activity                    | Budget  | Notes                                   |
|----------------------------|---------|-----------------------------------------|
| Event Partnerships          | $700    | Focus on 2-3 events; negotiate in-kind; limit onsite repair scope |
| Social Media Promotions     | $500    | $100 test; $400 scaled based on ROI     |
| Flyers & Printing           | $200    | Targeted small batches to avoid waste   |
| Google Basic Ads & SEO      | $300    | Prioritize Google My Business & review incentives, small ad tests |
| Contingency & Flexible Use  | $800    | Buffer for opportunity boosts, missed costs, or operational aids (e.g., scheduling apps) |

*Total:* $2,500

---

**4. Key Operational Considerations**

- **Workload Management:** Cap daily repair appointments; prioritize booked jobs over walk-ins post-events.
- **Onsite Event Services:** Limit to quick, value-adding fixes; funnel complex repairs to scheduled follow-ups.
- **Communication Efficiency:** Use templated replies and set clear response hours to manage inquiries promptly without overload.
- **Workshops & Partnerships:** Schedule during low-demand periods to avoid peak traffic conflict.
- **Performance Monitoring:** Track leads, bookings, and workload daily; be ready to scale marketing cautiously or pause based on capacity.
- **Customer Experience:** Maintain repair quality and turnaround times to preserve reputation and encourage repeat business.
- **Contingency Usage:** Utilize buffer to address unexpected expenses or invest in tools/apps to automate scheduling or customer communication.

---

This strategy balances aggressive local marketing with realistic operational capacity and budget constraints, positioning the business for steady awareness growth and revenue increase over the next 3 months without risking burnout or service decline.

# Discussion
Compare the single-agent baseline against the collaborative multi-agent strategy.

## Questions To Consider

- Did the specialized agents contribute meaningfully different perspectives?
- Did operational and financial constraints improve realism?
- Did the collaboration produce a more practical strategy?
- Which agent added the most value?
- Did any recommendations conflict with each other?
- Did the final strategy evolve over time rather than appearing all at once?
- Did the agents introduce useful tradeoffs or just generate more text?

---

## Notable Differences Between the Baseline and Multi-Agent Versions

### 1. The collaborative version introduced phased planning

The single-agent baseline mostly produced a static list of recommendations.

The collaborative version naturally evolved into a staged rollout:
- Month 1 setup
- Month 2 execution
- Month 3 optimization and scaling

This emerged from the interaction between agents rather than from explicit programming.

---

### 2. Operational realism improved significantly

The collaborative version repeatedly accounted for:
- workload limits
- scheduling constraints
- customer throughput
- burnout risk
- appointment management

This happened because OperationsAgent continuously pressured the plan toward feasibility.

Example:
> "Limit onsite event work to quick fixes and funnel larger repairs into scheduled follow-ups."

That type of operational refinement was largely absent from the baseline.

---

### 3. The system became adaptive instead of static

The baseline allocated most spending immediately.

The collaborative version introduced:
- test budgets
- incremental scaling
- monitoring
- adjustment over time

Example:
> Run a small ad campaign first, then scale spending based on results.

This is a much more realistic business-planning pattern.

---

### 4. Financial caution and uncertainty management emerged naturally

The collaborative version introduced:
- contingency reserves
- staged investment
- flexible budget allocation
- ROI-focused prioritization

The baseline optimized for full budget utilization.  
The collaborative version optimized for resilience and adaptability.

This emerged largely from FinanceAgent's influence.

---

### 5. The final strategy became more internally coordinated

The multi-agent version connected:
- marketing activities
- operational capacity
- budget constraints
- timing
- customer management

The baseline produced reasonable ideas, but the collaborative version produced a more integrated operational strategy.

---

## Important Point

The multi-agent system did not become "magically smarter."

The improvement came from:
- specialization
- critique
- refinement
- coordination between perspectives
- iterative planning under constraints

The agents created pressure on each other's assumptions, which improved the final artifact.

---

## Tradeoffs and Costs

The collaborative system also introduced costs:
- more complexity
- longer prompts
- larger context windows
- higher token usage
- more orchestration overhead

Multi-agent systems can improve outcomes, but only when:
- the agents have meaningful specialization
- the workflow is structured carefully
- the interaction improves the artifact rather than creating noise


## Key Takeaways

Collaborative multi-agent systems can improve planning quality through:
- specialization
- iterative refinement
- constraint-aware critique
- synthesis across perspectives

However, collaboration also introduces:
- orchestration complexity
- larger context windows
- higher token costs
- coordination challenges

Adding more agents does not automatically improve outcomes.

The workflow and interaction structure matter enormously.

---

## Relationship to Workflow Systems

This demo begins to resemble structured workflow orchestration:
- multiple specialized roles
- sequential refinement
- artifact evolution
- synthesis stages

As systems become larger and more complex, frameworks like LangGraph become increasingly valuable for:
- routing
- state management
- retries
- checkpoints
- human approvals
- deterministic execution

In practice, conversational agent systems and workflow systems are often combined together.